In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 43.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00


In [3]:
import zipfile
import os
import numpy as np
import cv2
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, pipeline

In [4]:
#распаковка всех файлов из архива
with zipfile.ZipFile('pictures_iad.zip', 'r') as zip_ref:
    zip_ref.extractall('output_directory')

In [5]:
image_paths = []

for root, dirs, files in os.walk("output_directory"):
    if "__MACOSX" in root:
        continue #игнорируем

    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(os.path.join(root, file))

print("Найдено изображений:", len(image_paths))
print(image_paths[:5])

Найдено изображений: 6
['output_directory/IMG_2695.jpg', 'output_directory/IMG_99998.jpg', 'output_directory/IMG_2696.jpg', 'output_directory/IMG_99999.jpg', 'output_directory/IMG_2694.jpg']


In [6]:
def to_rgb(image):
    return image.convert("RGB")

In [7]:
def smart_resize_for_vlm(image, target_size=512):
    """
    Сохраняем соотношение сторон, добавляем паддинг
    для квадратного изображения
    """
    width, height = image.size #исходные ширина и высота картинки
    max_side = max(width, height) #находим большую сторону
    scale = target_size / max_side #масштабируем так, чтобы максимальная сторона была target_size
    new_width = int(width * scale)
    new_height = int(height * scale)

    image = image.resize((new_width, new_height), Image.Resampling.LANCZOS) #ресайзим с сохранением пропорций

    new_image = Image.new('RGB', (target_size, target_size), (255, 255, 255)) #создаем новое квадратное изображение с белым фоном

    offset = ((target_size - new_width) // 2, (target_size - new_height) // 2) #вставляем наше изображение по центру
    new_image.paste(image, offset)

    return new_image

In [8]:
def remove_shadows_and_enhance(image_array):
    """
    Убираем тени и выравниваем освещение
    без потери цветовой информации
    """
    lab = cv2.cvtColor(image_array, cv2.COLOR_RGB2LAB) #конвертируем в LAB цветовое пространство (L - яркость, A и B - цвет)
    l, a, b = cv2.split(lab) #разделяем каналы
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)) #применяем CLAHE к каналу яркости (улучшает контрастность)
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b)) #собираем изображение обратно
    enhanced = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB) #конвертируем обратно в RGB

    return enhanced

In [9]:
def preprocessing(image):
    image = to_rgb(image)
    image = smart_resize_for_vlm(image, 512)
    image = remove_shadows_and_enhance(np.array(image))
    return Image.fromarray(image)

In [10]:
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear4bit(in_features=1280, out_features=3840, bias=True)
            (proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear4bit(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear4bit(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear4bit(in_features=3420, out_features=1280, bias=True

In [38]:
prompt = """Ты — система распознавания рукописного текста (OCR).

Твоя задача: точно переписать весь текст, который написан от руки на изображении.

Правила:
- Выводи ТОЛЬКО текст с изображения.
- Не описывай изображение.
- Не объясняй, что ты видишь.
- Не добавляй комментарии или интерпретации.
- Не исправляй орфографию и ошибки.
- Сохраняй оригинальное написание текста.
- Сохраняй переносы строк, если они есть.
- Если слово неразборчиво, попробуй угадать наиболее вероятный вариант по контексту.
- Если текст отсутствует или не читается, верни пустую строку.

Будь максимально точным и буквальным."""

results = []

for i, path in enumerate(image_paths):
    image = Image.open(path) #открываем изображение
    image = preprocessing(image) #выполняем предобработку

    messages = [
        {
            "role": "user", #сообщение от пользователя
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False, #на выходе - текст
        add_generation_prompt=True
    )

    image_inputs = [image]

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=5
        )

    generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]

    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

    results.append({"path": path, "prediction": output_text.strip()})
    print(f"{i+1}/{len(image_paths)} done - {os.path.basename(path)}")

1/6 done - IMG_2695.jpg
2/6 done - IMG_99998.jpg
3/6 done - IMG_2696.jpg
4/6 done - IMG_99999.jpg
5/6 done - IMG_2694.jpg
6/6 done - IMG_2697.jpg


In [17]:
!pip install cer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.2 MB/s eta 0:00:00


In [39]:
hyps = []

for res in results:
    text = res['prediction']
    hyps.append(text)

hyps

['Программирование - это не просто технические навыки, а особый способ мышления, позволяющий человеку создавать новые цифровые миры. В ХХI веке оно стало универсальным умом, взаимодействующим с механикой. Подобно тому как когда-то знание грамоты открывало доступ к культуре и науке, сегодня умение писать код открывает пути к инновациям, автоматизации и творчеству.',
 'Вот весь текст, который написан от руки на изображении:\n\nСтолько совести не хватит и тако,\nсколько мы едим, а им тако, как они едим!\nТак и едимъ, так и едимъ, лёвушка,\nэто вовсе не зависти аи дёсна\nв наших белях, которые мы украсим у\nжизни. Аи зависти только мы намело\nотношение к ним! Об этом сказано ещё в\nдамской этике: "кто умеет\nдовольствоваться, тому всегда будет\nдоволен."',
 'История Петербургского университета\nИстория Петербургского университета\nИстория Петербургского университета\nИстория Петербургского университета\nИстория Петербургского университета\nИстория Петербургского университета\nИстория Петер

In [40]:
# В hyps поместите выводы вашей модели, можете копирнуть руками, можете скриптом пройтись циклом по фото в датасете

from cer import calculate_cer_corpus


hyps = []

for res in results:
    text = res['prediction']
    hyps.append(text)

refs = ["""Современные языки программирования, такие как Python, Java и C++, позволяют решать широкий спектр задач — от создания веб-сайтов до разработки сложных систем искусственного интеллекта. Каждый язык обладает своей философией и областью применения, но все они основаны на общих принципах: алгоритмах, структурах данных и логике.""",
"""Программирование — это не просто технический навык, а особый способ мышления, позволяющий человеку создавать новые цифровые миры. В XXI веке оно стало универсальным языком взаимодействия с технологиями. Подобно тому как когда-то знание грамоты открывало доступ к культуре и науке, сегодня умение писать код открывает путь к инновациям, автоматизации и творчеству.""",
"""История программирования начинается задолго до появления современных компьютеров. Идеи алгоритмов можно проследить ещё в работах математиков прошлого, а в XIX веке Ада Лавлейс создала описание алгоритма для аналитической машины, став первым в мире программистом. С развитием вычислительной техники программирование превратилось в самостоятельную область знаний, объединяющую математику, инженерию и логику.""",
"""Это первая домашка на ИАДе.
Она, кажется, простая, но пока никто её не решил...""",
"""Сытость совсем не зависит от того,
сколько мы едим, а от того, как мы едим!
Так и счастье, так и счастье, Лёвушка,
оно вовсе не зависит от объёма
внешних благ, которые мы урвали у
жизни. Оно зависит только от нашего
отношения к ним! Об этом сказано ещё в
даосской этике: «Кто умеет
довольствоваться, тот всегда будет
доволен.""",
"При наличии уважительной причины дедлайн по д/з может быть перенесён. Дедлайн по д/з переносится на кол-во дней, равное продолжительности ув. причины."]

hyps = [sent.split() for sent in hyps]
refs = [sent.split() for sent in refs]

cer_corpus_score = calculate_cer_corpus(hyps, refs)
cer_corpus_score

{'count': 6,
 'mean': 0.8483481891931257,
 'median': 0.8278869521388756,
 'std': 0.08729761365466927,
 'min': 0.7659574468085106,
 'max': 1.0,
 'cer_scores': [0.8397731744839182,
  0.8160007297938332,
  0.7758846657929227,
  0.7659574468085106,
  0.8924731182795699,
  1.0]}

In [41]:
print(3 * min(1, 1 - cer_corpus_score["mean"] + 0.19))

1.024955432420623


**Источники:**
1. https://sky.pro/wiki/media/raspakovka-zip-fajlov-v-python/
2. https://sky.pro/wiki/media/iteracziya-po-fajlam-v-zadannoj-direktorii-v-python/
3. https://ai-manual.ru/article/vlm-slomalis-na-vashih-skanah-pochemu-multimodalki-ne-chitayut-tekst-i-kak-eto-ispravit-za-2-chasa/
4. https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct
5. https://colab.research.google.com/#scrollTo=RY6qpQ7v9-jA&fileId=https%3A//huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct.ipynb